# Identificação do projeto

**Título do projeto:** Perfil das faixas populares nos charts da Deezer por gênero  
**Integrantes do grupo:** Eduardo Fabri, João Correa, Rodrigo Alves e João Pedro Cardoso  
**Turma:** Bacharelado em Sistemas de Informação • 2º semestre de 2026  
**Descrição breve do problema ou contexto:** os charts da Deezer dizem quais faixas estão no topo de cada gênero, mas não o que elas têm em comum. Este notebook usa o conjunto preparado no RA1 para descrever e comparar, entre gêneros, a duração das faixas, a idade do lançamento, a popularidade do artista e a concentração de artistas no ranking, e testa uma previsão simples com Machine Learning.

---

# RA3 — Analysis, Visualization and Machine Learning

Utilize os dados preparados e processados nas etapas anteriores para produzir análises estatísticas, visualizações e interpretações que contribuam para responder ao problema definido. Explore também uma aplicação simples de Machine Learning, por classificação ou regressão, para investigar uma possibilidade de previsão a partir dos dados.

## 1. Análise estatística descritiva

Realize uma análise estatística dos dados, contemplando:

- **Tendência central:** utilização de média e/ou mediana para compreender valores representativos.
- **Dispersão:** utilização de medidas como amplitude ou desvio-padrão para analisar a variabilidade.
- **Distribuição:** investigação da frequência, concentração ou distribuição dos valores.

As medidas utilizadas devem ser coerentes com as características dos dados e com o problema investigado. Apresente os resultados e explique o que eles representam no contexto do projeto.

In [ ]:
# Preparação do ambiente: bibliotecas, pastas do projeto e origem dos dados
import subprocess
import sys
from pathlib import Path

import pandas as pd

URL_REPOSITORIO = "https://github.com/Jcliz/machine-learning-ra3.git"
PASTA_CLONE = "machine-learning-ra3"
RAMO = "main"  # no Colab, troque pelo ramo do repositório que quiser executar
NO_COLAB = "google.colab" in sys.modules

if NO_COLAB:
    if Path(PASTA_CLONE).exists():
        # Clone de uma execução anterior nesta sessão do Colab: traz a versão mais nova do ramo
        subprocess.run(["git", "-C", PASTA_CLONE, "pull", "-q", "origin", RAMO], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "-b", RAMO, URL_REPOSITORIO], check=True)

# O conjunto preparado fica em dados/ no repositório da entrega e em dados/preparados/ no de desenvolvimento
candidatos = [Path(PASTA_CLONE), Path.cwd(), Path.cwd().parent]
encontrados = [
    (p.resolve(), arquivo.resolve())
    for p in candidatos if (p / "requirements.txt").exists()
    for arquivo in (p / "dados" / "faixas_preparadas.csv", p / "dados" / "preparados" / "faixas_preparadas.csv")
    if arquivo.exists()
]
if not encontrados:
    raise FileNotFoundError(
        "Conjunto preparado não encontrado (dados/faixas_preparadas.csv ao lado do requirements.txt). "
        f"Rode o notebook de dentro do clone (raiz ou notebook/) ou, no Colab, confira o ramo '{RAMO}'."
    )
RAIZ, ARQUIVO_PREPARADO = encontrados[0]
PASTA_PREPARADOS = ARQUIVO_PREPARADO.parent

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
print(f"Raiz do projeto: {RAIZ.name}")  # só o nome da pasta: o caminho completo é da máquina de quem executa
print(f"Ambiente: {'Google Colab' if NO_COLAB else 'local'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

In [ ]:
# Releitura do conjunto preparado para a análise
colunas_inteiras_analise = ["id_faixa", "id_chart", "posicao_chart", "id_artista", "id_album",
                            "duracao_s", "rank", "numero_fas"]
colunas_texto_analise = ["titulo", "genero", "tipo_chart", "artista"]
colunas_booleanas_analise = ["lancamento_posterior_coleta", "duracao_fora_limites_iqr"]
df_analise = pd.read_csv(
    ARQUIVO_PREPARADO, encoding="utf-8",
    dtype={**{c: "int64" for c in colunas_inteiras_analise},
           **{c: "string" for c in colunas_texto_analise},
           **{c: "bool" for c in colunas_booleanas_analise}},
    parse_dates=["data_lancamento", "data_coleta"],
)
print(f"{len(df_analise)} ocorrências faixa-chart, {df_analise['id_faixa'].nunique()} faixas distintas, "
      f"{df_analise.loc[df_analise['tipo_chart'].eq('genero'), 'genero'].nunique()} charts de gênero "
      f"e {df_analise['tipo_chart'].eq('geral').sum()} ocorrências no chart geral.")

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

**Resposta do grupo:**

[Preencha aqui.]

## 2. Visualização e interpretação dos dados

Produza pelo menos três visualizações que contribuam para compreender diferentes características dos dados. As visualizações deverão:

- Ser adequadas às informações apresentadas.
- Possuir identificação clara das variáveis e dos resultados.
- Ser acompanhadas de uma breve interpretação.

Identifique padrões, tendências, diferenças ou comportamentos relevantes, evitando limitar a análise à simples apresentação de gráficos.

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

**Resposta do grupo:**

[Preencha aqui.]

## 3. Amostragem e representatividade

Apresente uma breve análise sobre a representatividade dos dados utilizados, abordando:

- A origem e a abrangência do conjunto de dados.
- Se os registros representam adequadamente o problema investigado.
- Possíveis limitações ou vieses que possam influenciar os resultados.

Não é obrigatório implementar um procedimento de amostragem. Caso utilize uma amostra, explique como os registros foram selecionados.

In [ ]:
# Origem e abrangência do conjunto
charts_repr = df_analise.groupby(["tipo_chart", "genero"]).size()
print(f"Data da coleta: {', '.join(df_analise['data_coleta'].dt.strftime('%d/%m/%Y').unique())}")
print(f"Ocorrências faixa-chart: {len(df_analise)} em {len(charts_repr)} charts "
      f"({charts_repr.loc['geral'].size} geral e {charts_repr.loc['genero'].size} de gênero), "
      f"de {charts_repr.min()} a {charts_repr.max()} ocorrências por chart")
print(f"Faixas distintas: {df_analise['id_faixa'].nunique()} · artistas distintos: {df_analise['id_artista'].nunique()} "
      f"· álbuns distintos: {df_analise['id_album'].nunique()}")
print(f"Posições no chart: de {df_analise['posicao_chart'].min()} a {df_analise['posicao_chart'].max()}")
lancamento_repr = df_analise["data_lancamento"]
print(f"Lançamentos: de {lancamento_repr.min():%d/%m/%Y} a {lancamento_repr.max():%d/%m/%Y}, "
      f"mediana em {lancamento_repr.median():%d/%m/%Y}")
print("Charts de gênero:", ", ".join(sorted(charts_repr.loc["genero"].index)))

In [ ]:
# Repetição de faixas entre charts
charts_por_faixa_repr = df_analise.groupby("id_faixa")["id_chart"].nunique()
repeticao_repr = charts_por_faixa_repr.value_counts().sort_index().rename_axis("charts por faixa").to_frame("faixas")
repeticao_repr["% das faixas"] = (100 * repeticao_repr["faixas"] / repeticao_repr["faixas"].sum()).round(1)
print(repeticao_repr.to_string())
faixas_geral_repr = set(df_analise.loc[df_analise["tipo_chart"].eq("geral"), "id_faixa"])
faixas_genero_repr = set(df_analise.loc[df_analise["tipo_chart"].eq("genero"), "id_faixa"])
print(f"\nFaixas do chart geral que também estão num chart de gênero: "
      f"{len(faixas_geral_repr & faixas_genero_repr)} de {len(faixas_geral_repr)}")

In [ ]:
# Diferenças de perfil entre os charts de gênero
generos_repr = df_analise[df_analise["tipo_chart"].eq("genero")].copy()
inicio_janela_repr = generos_repr["data_coleta"] - pd.DateOffset(years=1)
generos_repr["lancada_12_meses"] = (generos_repr["data_lancamento"].gt(inicio_janela_repr)
                                    & ~generos_repr["lancamento_posterior_coleta"])
perfil_repr = generos_repr.groupby("genero").agg(
    artistas_distintos=("id_artista", "nunique"),
    mediana_rank=("rank", "median"),
    mediana_fas=("numero_fas", "median"),
    pct_lancadas_12_meses=("lancada_12_meses", "mean"),
).sort_values("mediana_rank")
perfil_repr["pct_lancadas_12_meses"] = (100 * perfil_repr["pct_lancadas_12_meses"]).round(1)
print(perfil_repr.astype({"mediana_rank": "int64", "mediana_fas": "int64"}).to_string())
print(f"\nMediana do rank: {perfil_repr['mediana_rank'].min():,.0f} a {perfil_repr['mediana_rank'].max():,.0f} "
      f"entre os {len(perfil_repr)} charts de gênero".replace(",", "."))

In [ ]:
# Posição no chart e rank da faixa
correlacao_repr = df_analise.groupby("genero")[["posicao_chart", "rank"]].apply(
    lambda chart: chart["posicao_chart"].corr(chart["rank"], method="spearman"))
print(f"Correlação de Spearman entre posição e rank em cada um dos {len(correlacao_repr)} charts: "
      f"mediana {correlacao_repr.median():.2f}, de {correlacao_repr.min():.2f} a {correlacao_repr.max():.2f}")
print(f"Charts com correlação abaixo de -0,5 (posição alta = rank alto): {(correlacao_repr < -0.5).sum()}")
print(f"Rank: de {df_analise['rank'].min():,} a {df_analise['rank'].max():,}".replace(",", "."))
print(f"Ocorrências de artista com 0 fãs: {df_analise['numero_fas'].eq(0).sum()} de {len(df_analise)}")

**Resposta do grupo:**

**Origem e abrangência.** Os dados vêm da API pública da Deezer: um retrato único, coletado em 16/10/2026, com o chart geral e os 27 charts de gênero que a própria Deezer lista, cada um com as 100 primeiras posições, que é o máximo que a API devolve. São 2.800 ocorrências faixa-chart, 2.581 faixas, 1.688 artistas e 2.292 álbuns, com lançamentos de 1930 a 2026. Segundo a ficha da fonte do documento do RA1 (seção 02), a API respondeu como se a coleta viesse dos Estados Unidos, e o chart depende do país da requisição.

**Representam o problema?** Para o problema definido, o perfil das faixas no topo dos charts por gênero, sim: o conjunto é o próprio topo de cada chart naquele dia, sem amostragem, e todo chart tem as mesmas 100 posições, o que torna os gêneros comparáveis em tamanho. A repetição entre charts é baixa (2.394 faixas, 92,8%, aparecem num só chart) e 97 das 100 faixas do chart geral também estão num chart de gênero, por isso o geral fica à parte nas comparações. O conjunto não representa o catálogo da Deezer, a audiência de outros países nem a evolução no tempo.

**Limitações e vieses.** (1) Um só dia e um só país aparente: o retrato pode refletir lançamentos da semana, e as diferenças entre gêneros podem mudar em outra data. (2) Os gêneros são categorias editoriais da Deezer, e os charts de gênero diferem muito em popularidade: a mediana do `rank` vai de 73.231 (Jazz) a 830.084 (Pop), e a fatia de faixas lançadas nos 12 meses anteriores vai de 2,0% (Rock e Films/Games) a 57,0% (Country). Os charts apresentam perfis de popularidade diferentes. (3) O corte nas 100 primeiras posições restringe os valores, porque faltam as faixas pouco populares. Esse corte pode afetar a relação entre fãs e `rank` da P3 e o modelo da seção 4, mas o efeito não foi medido neste conjunto. (4) A posição no chart não segue o `rank`: a correlação de Spearman entre os dois tem mediana de -0,21 entre os 28 charts (de -0,53 a 0,32), e só 1 fica abaixo de -0,5. O `rank` mede a popularidade geral da faixa, e não a ordem do chart. (5) 8 ocorrências têm artistas com 0 fãs, valor mantido como veio da fonte.

**Amostragem.** Nenhuma etapa da análise sorteia registros: as seções 1 a 3 usam as 2.800 ocorrências. A única separação é a de treino e teste da seção 4, descrita lá.

## 4. Aplicação de Machine Learning

Implemente uma aplicação simples de aprendizagem supervisionada, por **classificação** (prever uma categoria ou classe) ou **regressão** (estimar um valor numérico). A implementação deverá contemplar:

1. Definição do problema preditivo.
2. Identificação da variável-alvo (*target*) e das características utilizadas (*features*).
3. Separação dos dados para treinamento e teste.
4. Treinamento de um modelo e realização de previsões.
5. Avaliação dos resultados utilizando pelo menos uma métrica adequada ao problema.

Caso o conjunto de dados original não possua características adequadas para uma aplicação supervisionada, outro conjunto de dados pode ser utilizado exclusivamente nesta etapa, com uma breve justificativa.

**Resposta do grupo:**

[Informe o problema preditivo, a variável-alvo e as características utilizadas.]

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

**Resposta do grupo:**

[Explique brevemente a métrica obtida e interprete o desempenho do modelo.]

## 5. Conclusões do RA3

Apresente uma conclusão geral do RA3, contemplando:

- Principais resultados identificados nas análises estatísticas e nas visualizações.
- Interpretação do desempenho do modelo de Machine Learning.
- Limitações dos dados e das análises realizadas.
- Relação dos resultados com o problema definido no projeto.

**Resposta do grupo:**

[Preencha aqui.]